<a href="https://colab.research.google.com/github/s251043276-tech/mantap/blob/main/Template_GitHub_Pull_Work_Push.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# @title 📂 Step 1: PULL (Download from GitHub)

import os
import shutil
import requests
import zipfile
from google.colab import userdata
from rpy2.robjects import r

# ============================================================
# 0A. FULL R RESET (NEW)
# ============================================================
print("🧽 Resetting R environment...")

r("""
# Close all graphics devices
if (!is.null(dev.list())) dev.off()

# Detach all non-base packages
base_pkgs <- c("stats","graphics","grDevices","utils","datasets","methods","base")
attached <- search()
for (pkg in attached) {
    if (grepl("^package:", pkg)) {
        pkg_name <- sub("^package:", "", pkg)
        if (!(pkg_name %in% base_pkgs)) {
            try(detach(pkg, unload=TRUE, character.only=TRUE), silent=TRUE)
        }
    }
}

# Clear global environment
rm(list = ls(all.names = TRUE))

# Clear hidden environments
try(rm(list = ls(envir = .GlobalEnv), envir = .GlobalEnv), silent=TRUE)

# Reset options
options(warn = 0)
options(stringsAsFactors = FALSE)

# Reset working directory
setwd("/content")

cat("✅ R environment fully reset.\n")
""")

# ============================================================
# 0B. CLEAN PROJECT DIRECTORY (NEW)
# ============================================================
PROJECT_PATH = "/content/project"

print("🧹 Cleaning /content/project ...")

if os.path.exists(PROJECT_PATH):
    shutil.rmtree(PROJECT_PATH)

os.makedirs(PROJECT_PATH, exist_ok=True)

print("✅ Clean slate ready at /content/project\n")

# ============================================================
# 1. STUDENT INPUTS
# ============================================================
GITHUB_ORG = ""       # @param {type:"string"}
GITHUB_USERNAME = ""  # @param {type:"string"}
REPO_NAME = ""       # @param {type:"string"}
STUDENT_EMAIL = ""  # @param {type:"string"}

# ============================================================
# 2. LOAD GITHUB PAT
# ============================================================
try:
    GITHUB_PAT = userdata.get("github")
except:
    raise RuntimeError("❌ ERROR: Add your GitHub PAT to Colab Secrets (key: 'github').")

# Determine repo owner
REPO_OWNER = GITHUB_ORG.strip() if GITHUB_ORG.strip() else GITHUB_USERNAME.strip()
clone_url = f"https://{GITHUB_PAT}@github.com/{REPO_OWNER}/{REPO_NAME}.git"

print(f"📛 Using repo owner: {REPO_OWNER}")

# ============================================================
# 3. AUTHENTICATED REPO EXISTENCE CHECK
# ============================================================
api_url = f"https://api.github.com/repos/{REPO_OWNER}/{REPO_NAME}"
headers = {"Authorization": f"token {GITHUB_PAT}"}

check = requests.get(api_url, headers=headers)

if check.status_code == 404:
    raise RuntimeError(
        f"❌ The repository '{REPO_OWNER}/{REPO_NAME}' does NOT exist.\n"
        "➡️ Create your own repo using any template you like.\n"
        "Then rerun this cell."
    )
elif check.status_code != 200:
    raise RuntimeError(
        f"❌ GitHub API error: {check.status_code}\n"
        "This is usually rate‑limit or authentication related.\n"
        "Try again shortly."
    )
else:
    print("✅ Repository exists on GitHub.")

# ============================================================
# 4. INITIALIZE R LIBRARY (only if missing)
# ============================================================
R_LIB_PATH = "/content/R_lib"
R_LIB_ZIP_URL = "https://github.com/ckyeoh-slides/r-libs-colab/releases/download/latest/R_lib.zip"

if not os.path.exists(R_LIB_PATH):
    print("📦 R library missing — downloading...")
    rlib_zip = "/content/R_lib.zip"

    with open(rlib_zip, "wb") as f:
        f.write(requests.get(R_LIB_ZIP_URL).content)

    print("🗜️ Extracting R library...")
    with zipfile.ZipFile(rlib_zip, "r") as z:
        z.extractall("/content")

    print("✅ R library ready!")
else:
    print("📚 R library already exists — skipping download.")

# ============================================================
# 5. INITIALIZE R ENVIRONMENT
# ============================================================
print("\n⚙️ Initializing R environment...")
%load_ext rpy2.ipython

r("rm(list = ls(all.names = TRUE))")
%R -i R_LIB_PATH

r(f"""
.libPaths(c("{R_LIB_PATH}", .libPaths()))
suppressPackageStartupMessages({{
    library(tidyverse)
    library(plotly)
    library(qcc)
    library(htmlwidgets)
}})
cat("📚 R libraries loaded from: {R_LIB_PATH}\n")
""")

# ============================================================
# 6. SYNC PROJECT (Clone fresh)
# ============================================================
print(f"\n🚀 Cloning fresh copy of {REPO_NAME}...")
!git clone {clone_url} {PROJECT_PATH}

# Git identity
!git config --global user.email "{STUDENT_EMAIL}"
!git config --global user.name "{GITHUB_USERNAME}"

# ============================================================
# 7. LOAD CSV DATA INTO R (Preview)
# ============================================================
print("\n📊 Loading CSV data into R...")

r_load_and_preview_script = """
setwd("/content/project")
data_path <- "data/"
if (dir.exists(data_path)) {
    files <- list.files(path = data_path, pattern = "\\\\.(csv|CSV)$")
    if (length(files) > 0) {
        for (f in files) {
            obj_name <- make.names(gsub("\\\\.(csv|CSV)$", "", f))
            df <- read.csv(paste0(data_path, f), check.names = TRUE)
            assign(obj_name, df, envir = .GlobalEnv)
            cat(paste0("\\n✅ Loaded: ", obj_name, "\\n"))
            cat("--------------------------------------------\\n")
            print(head(df, 3))
            cat("--------------------------------------------\\n")
        }
    } else {
        cat("📁 No CSV files found in data/.\\n")
    }
} else {
    cat("📁 data/ folder not found.\\n")
}
"""
r(r_load_and_preview_script)

print(f"\n✨ SUCCESS! Environment ready and synced to /content/project")
%cd /content


In [ ]:
# @title 🤖 AI Data Context Exporter
# @markdown Run this, copy the output, and paste it into your AI.
%%R
setwd("/content/project")

# 1. Identify all dataframes and metadata
df_list <- names(which(unlist(eapply(.GlobalEnv, is.data.frame))))

# Safe Read for Slides (Handles empty or missing files)
slides_content <- if(file.exists("slides.md") && file.info("slides.md")$size > 0) {
    readChar("slides.md", file.info("slides.md")$size)
} else { "Empty or New File" }

# Safe Read for Bibliography
bib_content <- if(file.exists("references.bib") && file.info("references.bib")$size > 0) {
    readChar("references.bib", file.info("references.bib")$size)
} else { "Missing or Empty" }

# Extract citation keys (CITATION_KEYS)
CITATION_KEYS <- if (bib_content != "Missing or Empty") {
    # find lines starting with @article{key, ...} or any @type{key,
    raw_keys <- regmatches(
        bib_content,
        gregexpr("@[A-Za-z]+\\{[^,]+", bib_content)
    )[[1]]
    # extract the part after '{'
    gsub(".*\\{", "", raw_keys)
} else character(0)

# 2. Header: Presentation & Technical Specs
cat("--- SYSTEM CONTEXT FOR AI ---
R is used ONLY for plots. Use %%R cells.

RULES — R:
- Use %%R only.
- No Python-to-R execution.
- No r(), no rpy2, no Python strings with R code.
- No assigning R code to Python vars.
- All R code lives directly inside %%R.

RULES — PYTHON:
- Python allowed for anything EXCEPT running R.

PATHS & EXPORT:
- Read CSVs from data/
- Save Plotly widgets to media/plots/<name>.html (saveWidget, selfcontained=TRUE)
- Save R code to media/plots/<name>.md (writeLines)

COLORS (Okabe-Ito):
- #0072B2, #D55E00, #009E73, #CC79A7

TYPOGRAPHY & MATH:
- Axis titles 18px, ticks 14px, white background
- Use LaTeX ($\\sigma$, $\\epsilon$)

SLIDES (Reveal.js):
- 2-column Pandoc layout: text LEFT, media RIGHT
- Plots: <iframe data-src='media/plots/<file>.html' width='100%' height='500px' style='border:none;'></iframe>
- Images: ![](media/pics/<file>.png)

CITATIONS:
- Use [@key]
- If a key is not in CITATION_KEYS, tell me to add it
\n")

cat("CITATION_KEYS:", paste(CITATION_KEYS, collapse=", "), "\n")
cat("\n")

cat("INSTRUCTIONS FOR YOUR RESPONSE:
R PLOTS:
- Return a COMPLETE Colab cell starting with %%R.
- It must: generate the plot, save the .html widget to media/plots/, and save the R code to media/plots/<name>.md.

SLIDES:
- Return a COMPLETE Python Colab cell.
- It must APPEND to slides.md using: with open('slides.md','a',encoding='utf-8') as f: f.write('''[NEW CONTENT]''')
------------------------------------------------------------

")

# 3. Loop through all datasets to provide schema and preview
if (length(df_list) > 0) {
    for (df_name in df_list) {
        target_df <- get(df_name)
        cat(paste0("DATASET: '", df_name, "'\n"))
        cat("==========================================\n")
        cat("PREVIEW (First 3 Rows):\n")
        cat(capture.output(write.csv(head(target_df, 3), row.names = FALSE)), sep="\n")
        cat("\nSCHEMA:\n")
        schema <- data.frame(
          Column = names(target_df),
          Type   = sapply(target_df, function(x) class(x)[1]),
          Example = sapply(target_df[1, ], function(x) substr(as.character(x), 1, 50))
        )
        print(schema, row.names = FALSE)
        cat("\n")
    }
} else {
    cat("❌ No dataframes found. Run Step 1 first.\n")
}

# --- CURRENT SLIDES (delimited, no markdown fences) ---
cat("CURRENT_SLIDES_START\n")
cat(slides_content)
cat("\nCURRENT_SLIDES_END\n\n")
cat("USER_REQUEST:\n\n")

In [ ]:
# @title 📤 Step 2: PUSH (Upload to GitHub)
commit_message = ""  # @param {type:"string"}

import os
from datetime import datetime
import pytz
from google.colab import userdata

# 1. Path & Identity Setup
PROJECT_PATH = "/content/project"
GITHUB_PAT = userdata.get('github')

# Robust variable checking
try:
    r_owner = GITHUB_ORG.strip() if 'GITHUB_ORG' in locals() and GITHUB_ORG.strip() else GITHUB_USERNAME
    r_name = REPO_NAME if 'REPO_NAME' in locals() else "slides-repo"
    PAGES_URL = f"https://{r_owner}.github.io/{r_name}/"
except NameError:
    raise NameError("❌ REPO_NAME or GITHUB_USERNAME not defined. Run Step 1 first!")

if os.path.exists(PROJECT_PATH):
    %cd {PROJECT_PATH}

    # --- PRIVACY FIX ---
    !git config --global user.email "{GITHUB_USERNAME}@users.noreply.github.com"
    !git config --global user.name "{GITHUB_USERNAME}"

    # --- MODERN PANDOC INSTALLATION ---
    pandoc_check = !pandoc --version
    if not any("3." in line for line in pandoc_check):
        print("📥 Installing Modern Pandoc (v3.1.1)...")
        !wget -q https://github.com/jgm/pandoc/releases/download/3.1.1/pandoc-3.1.1-1-amd64.deb
        !dpkg -i pandoc-3.1.1-1-amd64.deb > /dev/null
        !rm pandoc-3.1.1-1-amd64.deb

    # --- PANDOC BUILD ---
    print("🚀 Building Reveal.js slides...")

    if os.path.exists("index.html"):
        os.remove("index.html")

    result = !pandoc -t revealjs -s "slides.md" -o "index.html" \
      --citeproc \
      --mathjax \
      -V revealjs-url=https://unpkg.com/reveal.js@4 \
      -V theme="serif" \
      -V transition=none \
      -V width='"90%"' \
      -V margin=0.1 \
      --include-in-header=custom-styles.html

    if os.path.exists("index.html"):
        print("✅ Build Successful. Syncing to GitHub...")

        # --- GIT LOGIC ---
        if os.path.exists(".git/index.lock"):
            os.remove(".git/index.lock")

        !git add .
        diff = !git status --porcelain

        if diff:
            # Auto‑generate MYT timestamp if commit message is blank
            final_msg = commit_message.strip()
            is_manual = bool(final_msg)

            if not final_msg:
                malaysia = pytz.timezone("Asia/Kuala_Lumpur")
                final_msg = f"auto‑save {datetime.now(malaysia).strftime('%Y-%m-%d %H:%M:%S')}"

            # Commit
            !git commit -m "{final_msg}"

            # Tag only meaningful commits
            if is_manual:
                !git tag -a checkpoint -m "checkpoint"

            # Push commit + tags
            remote_url = f"https://{GITHUB_PAT}@github.com/{r_owner}/{r_name}.git"
            push_output = !git push {remote_url} main --force
            !git push {remote_url} --tags

            if any("rejected" in line or "error" in line for line in push_output):
                print("❌ PUSH FAILED. Manual check required.")
                print("\n".join(push_output))
            else:
                print("\n" + "="*60)
                print(f"🎉 SUCCESS! Your work is saved.")
                print(f"🌐 VIEW YOUR LIVE PORTFOLIO: {PAGES_URL}")
                print("="*60)

        else:
            print("ℹ️ No changes detected. GitHub is already up to date.")

    else:
        print("❌ BUILD FAILED: Pandoc could not create index.html")
        print("\n".join(result))

    %cd /content

else:
    print(f"❌ ERROR: Project folder not found at {PROJECT_PATH}")


In [ ]:
# @title 🔨 WARNING!!! Hard Reset to a Specific Commit (Destructive)

import os

PROJECT_PATH = "/content/project"   # same path as your main workflow

# ============================================================
# 1. STUDENT INPUT: Commit hash to reset to
# ============================================================
TARGET_COMMIT = ""  # @param {type:"string"}

if TARGET_COMMIT.strip() == "":
    raise RuntimeError("❌ No commit hash provided. Paste a valid commit hash.")

# ============================================================
# 2. VERIFY PROJECT EXISTS
# ============================================================
if not os.path.exists(f"{PROJECT_PATH}/.git"):
    raise RuntimeError("❌ No Git repo found at /content/project. Run Step 1 first.")

# ============================================================
# 3. SHOW CURRENT HEAD (for context)
# ============================================================
print("📌 Current HEAD:")
%cd {PROJECT_PATH}
!git --no-pager log -1 --oneline

# ============================================================
# 4. PERFORM HARD RESET
# ============================================================
print(f"\n⚠️ Performing hard reset to commit: {TARGET_COMMIT}")
!git reset --hard {TARGET_COMMIT}

# ============================================================
# 5. FORCE PUSH TO REMOTE
# ============================================================
print("\n🚀 Force pushing rewritten history to remote...")
!git push --force

print("\n✨ Done. Repo history rewritten.")
%cd /content
